# Can one isotope trap a lattice vibration?
The springs are identical and a single mass differs from the rest. Predict which sign of mass change produces a mode above the propagation band. The chain is periodic so an edge cannot trap the vibration.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

mass_ratio = .60       # impurity mass / host mass; also try 1 or 1.4
sites = 80             # even, to include the host zone boundary on the mesh
assert .12<=mass_ratio<=2 and 12<=sites<=180 and sites%2==0

## Assemble the symmetric eigenproblem
Use host mass and spring constant as units. Converting from mass-weighted coordinates back to actual site displacement matters most at the impurity.

In [ ]:
def dynamical(n,r):
    K=2*np.eye(n)-np.roll(np.eye(n),1,axis=1)-np.roll(np.eye(n),-1,axis=1)
    mass=np.ones(n);mass[n//2]=r
    return K/np.sqrt(mass[:,None]*mass[None,:]),mass
D,mass=dynamical(sites,mass_ratio)
values,vec=np.linalg.eigh(D); w=np.sqrt(max(values[-1],0))
u=vec[:,-1]/np.sqrt(mass)
u/=np.max(abs(u))
if u[sites//2]<0:u=-u
weight=mass*u*u;weight/=weight.sum()
pos=np.arange(sites)-sites//2
print(f"Numerical frequency {w:.6f} in units of sqrt(K/M); impurity kinetic weight {weight[sites//2]:.4f}")

## Compare the spatial profile and the infinite-chain dispersion
The analytic profile alternates sign when the frequency lies above the band. No decaying above-band solution exists for a heavy defect; its uppermost finite-chain mode remains a valid numerical eigenvector.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(9,3.9),layout="constrained")
ax[0].plot(pos,u,"o-",ms=3,color="#175868",label="Periodic-chain eigenvector")
if mass_ratio<1:
    r=mass_ratio/(2-mass_ratio)
    ax[0].plot(pos,(-r)**abs(pos),"x",color="#a23d21",label="Infinite chain")
ax[0].set(xlabel="Site relative to impurity",ylabel="Relative displacement",title="Highest-frequency mode",
          xlim=(-min(12,sites//2),min(12,sites//2)));ax[0].legend(fontsize=8)
ratios=np.linspace(.22,1.8,48)
finite=[np.sqrt(np.linalg.eigvalsh(dynamical(min(48,sites),q)[0])[-1]) for q in ratios]
qs=np.linspace(.22,.999,200)
ax[1].plot(ratios,finite,".",color="#375b9a",label="Finite chain")
ax[1].plot(qs,2/np.sqrt(qs*(2-qs)),color="#a23d21",label="Localised solution")
ax[1].axhline(2,ls=":",color="#175868",label="Host band edge");ax[1].plot(mass_ratio,w,"o",color="#175868")
ax[1].set(xlabel="Impurity / host mass",ylabel="Frequency / √(K/M)",title="How the local mode detaches");ax[1].legend(fontsize=8)
plt.show()

## Test translation symmetry and the original force balance
Check the acoustic mode and the equation before mass weighting. Increase the ring size at a mass ratio just below one and track the small difference between the numerical eigenfrequency and the infinite-chain limit.

In [ ]:
assert abs(values[0])<1e-12
res=2*u-np.roll(u,1)-np.roll(u,-1)-w*w*mass*u
assert np.max(abs(res))<1e-10
if mass_ratio<1:
    analytic=2/np.sqrt(mass_ratio*(2-mass_ratio)); length=-1/np.log(mass_ratio/(2-mass_ratio))
    print(f"Infinite-chain frequency {analytic:.6f}, decay length {length:.3f} spacings")

## Optional independent OneSimulation eigensolve
Run the identical mass-weighted operator through the OneSimulation symmetric eigensystem interface, then check its highest eigenvalue and residual in the original coordinates.

In [ ]:
try:
    from onesim.numerics.linear.matrices.dense_linear_algebra import symmetric_eigensystem
except ImportError:
    print("The optional eigensolver check runs in a local OneSimulation installation.")
else:
    E,Q=symmetric_eigensystem(D)
    print("OneSimulation eigenvalue residual:",np.linalg.norm(D@Q[:,-1]-E[-1]*Q[:,-1]))
    assert np.isclose(E[-1],w*w)